# Single-Word MEG Decoding — Colab Pipeline

**One-time setup (do this before running):**
1. Open the shared Drive folder: https://drive.google.com/drive/folders/1HOvzJEv0Czn-yJKELq6kp5cORJ5yBTdM
2. Right-click it → **Organize** → **Add shortcut to Drive** → place it in **My Drive** and name it exactly `single-word-decoding`

**Each session:**
1. Run **Cell 1** (mount Drive, requires one click to authorize)
2. Run **Cell 2** (full setup, clone, install, data — walk away for ~10 min)
3. Run **Cell 3** (train)
4. Run **Cell 4** (check results)

**Data strategy:**
- First session: downloads from OSF → local SSD → backs up to shared Drive
- Later sessions: restores from shared Drive → local SSD (no re-download)
- Checkpoints always saved to shared Drive (survive session resets, visible to all teammates)

## Cell 1 — Mount Drive
Run this first. It will ask you to sign in and authorize access.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, torch

# ── Paths ──────────────────────────────────────────────────────────────
ALL_SUBJECTS   = [f'sub-{i:02d}' for i in range(1, 28)]  # all 27 subjects to back up to Drive
TRAIN_SUBJECTS = ['sub-01', 'sub-02', 'sub-03']           # subjects to restore locally for training
BATCH_SIZE     = 10                                        # subjects to download at a time (~70-80 GB per batch)

REPO           = 'https://github.com/kazumah1/single-word-decoding.git'
WORKDIR        = '/content/single-word-decoding'
LOCAL_DATAPATH = '/content/neural_data'
DRIVE_ROOT     = '/content/drive/MyDrive/single-word-decoding'
DRIVE_DATAPATH = f'{DRIVE_ROOT}/neural_data'
SAVEPATH       = f'{DRIVE_ROOT}/sentence_results'
# ───────────────────────────────────────────────────────────────────────

os.makedirs(f'{DRIVE_DATAPATH}/gwilliams2022', exist_ok=True)
os.makedirs(f'{SAVEPATH}/cache',               exist_ok=True)
os.makedirs(LOCAL_DATAPATH,                    exist_ok=True)

print(f"GPU:  {torch.cuda.get_device_name(0)}")
total, _, free = shutil.disk_usage('/content')
print(f"Disk: {free/1e9:.0f} GB free / {total/1e9:.0f} GB total")
print(f"\nAll subjects:   {ALL_SUBJECTS}")
print(f"Train subjects: {TRAIN_SUBJECTS}")
print(f"Paths set. Run Cell 2 to set up environment.")

## Cell 2 — Full Setup (run once per session, then walk away)
Clones repo, installs all dependencies, and gets data onto local disk.

In [ ]:
import os, shutil, subprocess, sys

# ── 1. Clone repo ──────────────────────────────────────────────────────
print('=== Cloning repo ===')
if os.path.exists(WORKDIR):
    print('  Already cloned, pulling latest...')
    subprocess.run(['git', '-C', WORKDIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO, WORKDIR], check=True)

os.chdir(WORKDIR)
if not os.path.exists(f'{WORKDIR}/sentence_results'):
    os.symlink(SAVEPATH, f'{WORKDIR}/sentence_results')
os.makedirs(f'{WORKDIR}/projects', exist_ok=True)
print('  Done.')

# ── 2. Install local packages ──────────────────────────────────────────
print('\n=== Installing local packages ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '--config-settings', 'editable_mode=strict', '-e', 'neuralset/'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '--config-settings', 'editable_mode=strict', '-e', 'neuraltrain/'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', 'sentence_decoding/'], check=True)
print('  Done.')

# ── 3. Install pinned dependencies ─────────────────────────────────────
print('\n=== Installing dependencies ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'lightning', 'pytorch-lightning', 'torchvision',
                'wandb', 'osfclient', 'mne_bids', 'tqdm'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'x-transformers==1.26.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchmetrics==1.5.2'], check=True)
print('  Done.')

# ── 4. kenlm ──────────────────────────────────────────────────────────
print('\n=== Installing kenlm ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kenlm'], capture_output=True)
try:
    import kenlm
    print('  Installed from PyPI.')
except ImportError:
    print('  PyPI failed — building from source...')
    subprocess.run(['git', 'clone', 'https://github.com/kpu/kenlm', '/content/kenlm'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'cython'], check=True)
    subprocess.run(['cython', '/content/kenlm/python/kenlm.pyx', '--cplus'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '/content/kenlm'], check=True)
    import kenlm
    print('  Built from source.')

# ── 5. Batch download all subjects to Drive ────────────────────────────
print('\n=== Downloading all subjects to Drive (batched) ===')
local_gw = f'{LOCAL_DATAPATH}/gwilliams2022/download'
drive_gw = f'{DRIVE_DATAPATH}/gwilliams2022/download'
os.makedirs(local_gw, exist_ok=True)
os.makedirs(drive_gw, exist_ok=True)

already_drive = [d for d in os.listdir(drive_gw) if d.startswith('sub-')] if os.path.exists(drive_gw) else []
to_download   = [s for s in ALL_SUBJECTS if s not in already_drive]

print(f'  Already on Drive: {sorted(already_drive)}')
print(f'  Need to download: {to_download}')

# Process in batches: download → backup to Drive → clear local
for batch_start in range(0, len(to_download), BATCH_SIZE):
    batch = to_download[batch_start:batch_start + BATCH_SIZE]
    print(f'\n  --- Batch {batch_start // BATCH_SIZE + 1}: {batch} ---')

    _, _, free = shutil.disk_usage('/content')
    print(f'  Disk free before batch: {free/1e9:.0f} GB')

    subprocess.run(
        [sys.executable, 'download_gwilliams_subject.py',
         '--subjects', *batch,
         '--path', f'{LOCAL_DATAPATH}/gwilliams2022'],
        check=True
    )

    # Backup stimuli once
    for folder in ['stimuli']:
        if os.path.exists(f'{local_gw}/{folder}') and not os.path.exists(f'{drive_gw}/{folder}'):
            print(f'  Backing up {folder} to Drive...')
            shutil.copytree(f'{local_gw}/{folder}', f'{drive_gw}/{folder}')

    # Backup each subject then delete local copy to free space
    for subject in batch:
        if os.path.exists(f'{local_gw}/{subject}'):
            if not os.path.exists(f'{drive_gw}/{subject}'):
                print(f'  Backing up {subject} to Drive...')
                shutil.copytree(f'{local_gw}/{subject}', f'{drive_gw}/{subject}')
            print(f'  Clearing local {subject} to free disk...')
            shutil.rmtree(f'{local_gw}/{subject}')

    _, _, free = shutil.disk_usage('/content')
    print(f'  Disk free after batch: {free/1e9:.0f} GB')

print(f'\n  All subjects backed up to Drive.')

# ── 6. Restore only TRAIN_SUBJECTS to local for training ──────────────
print('\n=== Restoring training subjects to local disk ===')
for subject in TRAIN_SUBJECTS:
    if not os.path.exists(f'{local_gw}/{subject}'):
        if os.path.exists(f'{drive_gw}/{subject}'):
            print(f'  Restoring {subject} from Drive...')
            shutil.copytree(f'{drive_gw}/{subject}', f'{local_gw}/{subject}')
        else:
            print(f'  WARNING: {subject} not found on Drive, downloading...')
            subprocess.run(
                [sys.executable, 'download_gwilliams_subject.py',
                 '--subjects', subject,
                 '--path', f'{LOCAL_DATAPATH}/gwilliams2022'],
                check=True
            )
    else:
        print(f'  {subject} already on local disk.')

for folder in ['stimuli']:
    if os.path.exists(f'{drive_gw}/{folder}') and not os.path.exists(f'{local_gw}/{folder}'):
        print(f'  Restoring {folder}...')
        shutil.copytree(f'{drive_gw}/{folder}', f'{local_gw}/{folder}')

# ── 7. Verify patches ─────────────────────────────────────────────────
print('\n=== Verifying code patches ===')
checks = [
    ('UninitializedParameter', 'sentence_decoding/main.py'),
    ('inference_mode',         'sentence_decoding/main.py'),
    ('Gwilliams2022',          'sentence_decoding/grids/test.py'),
]
for pattern, filepath in checks:
    r = subprocess.run(['grep', '-l', pattern, filepath], capture_output=True, text=True)
    status = '[OK]' if r.returncode == 0 else '[MISSING]'
    print(f'  {status} {pattern} in {filepath}')

print('\n✓ Setup complete. Run Cell 3 to start training.')

## Cell 3 — Train

In [ ]:
import os
os.environ['DATAPATH']   = LOCAL_DATAPATH
os.environ['SAVEPATH']   = SAVEPATH
os.environ['WANDB_MODE'] = 'disabled'  # set to 'online' to enable wandb

!DATAPATH={LOCAL_DATAPATH} SAVEPATH={SAVEPATH} WANDB_MODE=disabled \
    python -m sentence_decoding.grids.test

## Cell 4 — Check Results

In [ ]:
import glob, json, os

result_dirs = sorted(glob.glob(f'{SAVEPATH}/test/*'))
if not result_dirs:
    print('No results yet.')
else:
    latest = result_dirs[-1]
    print(f'Latest run: {latest}')
    for f in sorted(os.listdir(latest)):
        print(f'  {f}')

    metrics_path = os.path.join(latest, 'metrics.json')
    if os.path.exists(metrics_path):
        with open(metrics_path) as fh:
            metrics = json.load(fh)
        print('\nMetrics:')
        for k, v in metrics.items():
            print(f'  {k}: {v}')